In [8]:
import os
import subprocess
from pathlib import Path
import pandas as pd
from typing import List, Optional

# =========================
# CONFIG (EDIT THESE PATHS)
# =========================
CLONE_ROOT = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Clone")
BOUNDARY_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_RQ5\List_Boundary_Commit_Events.csv")  # <-- change to where your csv is
OUTDIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_RQ5")
OUTDIR.mkdir(parents=True, exist_ok=True)

WORKFLOW_PATHS = [".github/workflows", ".github/main.workflow"]

# Git executable autodetect
GIT_EXE_CANDIDATES = [
    "git",
    r"C:\Program Files\Git\bin\git.exe",
    r"C:\Program Files\Git\cmd\git.exe",
    r"C:\Program Files (x86)\Git\bin\git.exe",
    r"C:\Program Files (x86)\Git\cmd\git.exe",
]

def pick_git_exe() -> str:
    for exe in GIT_EXE_CANDIDATES:
        try:
            r = subprocess.run([exe, "--version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            if r.returncode == 0 and "git version" in (r.stdout.lower() + r.stderr.lower()):
                return exe
        except FileNotFoundError:
            continue
    raise FileNotFoundError("Could not find git.exe. Install Git for Windows or add it to PATH.")

GIT_EXE = pick_git_exe()
print("Using git:", GIT_EXE)

def run_git(repo_dir: Path, args: List[str]) -> subprocess.CompletedProcess:
    return subprocess.run(
        [GIT_EXE, "-C", str(repo_dir), *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

def is_git_repo(repo_dir: Path) -> bool:
    r = run_git(repo_dir, ["rev-parse", "--is-inside-work-tree"])
    return r.returncode == 0 and r.stdout.strip() == "true"

def commit_exists(repo_dir: Path, sha: str) -> bool:
    r = run_git(repo_dir, ["cat-file", "-e", f"{sha}^{{commit}}"])
    return r.returncode == 0

def actions_present_at_commit(repo_dir: Path, sha: str) -> Optional[bool]:
    """
    True  => at commit sha, at least one Actions workflow path exists
    False => commit exists, but no workflow paths exist at that commit
    None  => repo/commit missing locally or other git error
    """
    if not is_git_repo(repo_dir):
        return None
    if not commit_exists(repo_dir, sha):
        return None

    r = run_git(repo_dir, ["ls-tree", "-r", "--name-only", sha, "--", *WORKFLOW_PATHS])
    if r.returncode != 0:
        return None
    return bool(r.stdout.strip())

def first_actions_commit(repo_dir: Path) -> Optional[str]:
    r = run_git(repo_dir, ["rev-list", "--all", "--reverse", "-n", "1", "--", *WORKFLOW_PATHS])
    sha = r.stdout.strip() if r.returncode == 0 else ""
    return sha or None

def commit_iso_date(repo_dir: Path, sha: str) -> Optional[str]:
    r = run_git(repo_dir, ["show", "-s", "--format=%cI", sha])
    if r.returncode != 0:
        return None
    d = r.stdout.strip()
    return d if d else None

# =========================
# LOAD + PICK FIRST START BOUNDARY PER REPO
# =========================
df = pd.read_csv(BOUNDARY_CSV)

# Ensure types / sort keys
df["is_episode_start_boundary"] = df["is_episode_start_boundary"].astype(int)
df["episode_index"] = df["episode_index"].astype(int)
df["event_date_utc"] = pd.to_datetime(df["event_date_utc"], errors="coerce")

starts = df[df["is_episode_start_boundary"] == 1].copy()

# "first instrumentation episode" = min episode_index, tie-break by earliest date
starts = starts.sort_values(["repo_name", "episode_index", "event_date_utc"])
first_start = starts.groupby("repo_name", as_index=False).first()

print("Repos with at least one start boundary:", first_start["repo_name"].nunique())

# =========================
# CHECK ACTIONS PRESENCE AT THAT START COMMIT
# =========================
records = []
missing_repo_dirs = 0

for _, row in first_start.iterrows():
    repo_name = row["repo_name"]
    repo_dir = CLONE_ROOT / repo_name
    sha = str(row["commit_sha"]).strip()

    rec = {
        "repo_name": repo_name,
        "repo_dir": str(repo_dir),
        "full_name": row.get("full_name", ""),
        "first_episode_index": int(row["episode_index"]),
        "first_start_env_style": row.get("env_style", ""),
        "first_start_event_type": row.get("event_type", ""),
        "first_start_event_date_utc": str(row["event_date_utc"]) if pd.notna(row["event_date_utc"]) else "",
        "first_start_commit_sha": sha,
        "repo_found_locally": repo_dir.exists() and repo_dir.is_dir(),
        "is_git_repo": False,
        "actions_present_at_first_start_boundary": "",
        "first_actions_commit_in_history": "",
        "first_actions_date_in_history": "",
        "error": "",
    }

    if not rec["repo_found_locally"]:
        missing_repo_dirs += 1
        rec["error"] = "REPO_DIR_NOT_FOUND"
        records.append(rec)
        continue

    rec["is_git_repo"] = is_git_repo(repo_dir)
    if not rec["is_git_repo"]:
        rec["error"] = "NOT_A_GIT_REPO"
        records.append(rec)
        continue

    present = actions_present_at_commit(repo_dir, sha)
    if present is None:
        rec["actions_present_at_first_start_boundary"] = ""
        rec["error"] = "START_SHA_NOT_FOUND_LOCALLY_OR_GIT_ERROR"
    else:
        rec["actions_present_at_first_start_boundary"] = bool(present)

    fac = first_actions_commit(repo_dir)
    rec["first_actions_commit_in_history"] = fac or ""
    rec["first_actions_date_in_history"] = commit_iso_date(repo_dir, fac) if fac else ""

    records.append(rec)

out = pd.DataFrame(records)

# =========================
# WRITE OUTPUTS
# =========================
all_path = OUTDIR / "repos_first_episode_start_has_actions.csv"
out.to_csv(all_path, index=False)

true_subset = out[out["actions_present_at_first_start_boundary"] == True].copy()
true_path = OUTDIR / "repos_first_episode_start_has_actions_TRUE.csv"
true_subset.to_csv(true_path, index=False)

print("\nDone.")
print("Missing repo dirs:", missing_repo_dirs)
print("Total repos checked:", len(out))
print("Actions present at FIRST start boundary:", (out["actions_present_at_first_start_boundary"] == True).sum())
print("Wrote:\n -", all_path, "\n -", true_path)


Using git: git
Repos with at least one start boundary: 399

Done.
Missing repo dirs: 1
Total repos checked: 399
Actions present at FIRST start boundary: 201
Wrote:
 - C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_RQ5\repos_first_episode_start_has_actions.csv 
 - C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_RQ5\repos_first_episode_start_has_actions_TRUE.csv
